In [ ]:
# =============================================================================
# wave2d_cuda – Visualiser for Google Colab
# =============================================================================
# Paste each cell block into a separate Colab cell, or run top-to-bottom.
# =============================================================================


# ── Cell 1 ── Install / configure ────────────────────────────────────────────
# ffmpeg is needed to encode the animation to MP4 (already present on Colab,
# but this ensures it is available).
# matplotlib inline + rc tweak for crisp rendering inside Colab.

# %matplotlib inline          ← uncomment if running as a .py via nbconvert

import subprocess, shutil

if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=True)

import matplotlib
matplotlib.rcParams["animation.embed_limit"] = 50   # MB – raise if needed
matplotlib.rcParams["figure.dpi"]            = 100

In [ ]:
# ── Cell 2 ── Upload the binary snapshot file ─────────────────────────────────
# Run this cell to upload wave2d_snapshots.bin from your local machine.
# Skip and set FNAME manually if the file is already on Drive / already uploaded.

# from google.colab import files as _colab_files

# print("Upload wave2d_snapshots.bin …")
# uploaded = _colab_files.upload()                    # opens the file-picker dialog

# Grab the filename that was just uploaded (works even if renamed)
# FNAME = next(iter(uploaded))
FNAME = "wave2d_snapshots.bin"
print(f"Using file: {FNAME}")

In [ ]:
# ── Cell 3 ── Parse the binary file ───────────────────────────────────────────

import struct
import numpy as np

with open(FNAME, "rb") as f:
    nx, ny, n_snaps = struct.unpack("3i", f.read(12))
    print(f"Grid {nx}×{ny}, {n_snaps} snapshots")

    frames = []
    for _ in range(n_snaps):
        _step = struct.unpack("i", f.read(4))[0]
        data  = np.frombuffer(f.read(nx * ny * 4), dtype=np.float32).reshape(nx, ny)
        frames.append(data)

print("Frames loaded:", len(frames))

In [ ]:
# ── Cell 4 ── Build & display the animation ────────────────────────────────────
# Colab cannot call plt.show() for animations; we use:
#   • HTML5 video  →  rendered inline in the notebook  (recommended)
#   • or save as .mp4 / .gif to download

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

LENGTH = 10.0

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    frames[0].T,
    extent=[0, LENGTH, 0, LENGTH],
    origin="lower",
    vmin=-1, vmax=1,
    cmap="viridis",
    interpolation="bilinear",
)
cbar = fig.colorbar(im, ax=ax, label="Amplitude")
ax.set_xlabel("x")
ax.set_ylabel("y")
title = ax.set_title("2D Wave Propagation  (CUDA)  – frame 0")
fig.tight_layout()

def _update(i: int):
    im.set_array(frames[i].T)
    title.set_text(f"2D Wave Propagation  (CUDA)  – frame {i:04d}")
    return im, title

ani = animation.FuncAnimation(
    fig,
    _update,
    frames=len(frames),
    interval=90,        # ms between frames
    blit=True,
)

plt.close(fig)          # prevents a static duplicate image from appearing

# Render as an HTML5 <video> tag – plays directly in the Colab output cell
html_video = ani.to_html5_video()
display(HTML(html_video))

In [ ]:
# ── Cell 5 (optional) ── Save MP4 / GIF to disk and download ──────────────────
from google.colab import files


# --- MP4 ---
MP4_PATH = "wave2d.mp4"
ani.save(MP4_PATH, writer=animation.FFMpegWriter(fps=30, bitrate=1800))
print(f"Saved {MP4_PATH}")
files.download(MP4_PATH)

# --- GIF (smaller, but larger file size; comment out if not needed) ---
# GIF_PATH = "wave2d.gif"
# ani.save(GIF_PATH, writer=animation.PillowWriter(fps=20))
# print(f"Saved {GIF_PATH}")
# _colab_files.download(GIF_PATH)
